SuiteSparse `.mtx` → `boruvka/graphs`, затем сводка, предобработка, бенч `boruvka_spla` / `boruvka_spark`.

In [29]:
import os, ssl, tarfile, tempfile, urllib.request
from pathlib import Path

_SSL = ssl._create_unverified_context()
_CHUNK = 8 * 1024 * 1024
_TMO = 600
BASE = "https://suitesparse-collection-website.herokuapp.com/MM"
OUT = (Path("..") / "boruvka" / "graphs").resolve()
OUT.mkdir(parents=True, exist_ok=True)

# stem → (SuiteSparse group, matrix name)
GRAPHS = [
    ("amazon0601", "SNAP", "amazon0601"),
    ("citationCiteseer", "DIMACS10", "citationCiteseer"),
    # ("as-skitter", "SNAP", "as-Skitter"),  # Spark: error
    ("coPapersDBLP", "DIMACS10", "coPapersDBLP"),
    ("in-2004", "LAW", "in-2004"),
    # ("soc-LiveJournal1", "SNAP", "soc-LiveJournal1"),  # Spark: error
    ("cit-Patents", "SNAP", "cit-Patents"),
    # ("USA-road-d.USA", "DIMACS10", "road_usa"),  # Spark: timeout
    # ("delaunay_n24", "DIMACS10", "delaunay_n24"),  # Spark: timeout
    # ("kron_g500-logn21", "DIMACS10", "kron_g500-logn21"),  # Spark: timeout
    # ("europe_osm", "DIMACS10", "europe_osm"),  # Spark: timeout
    # ("uk-2002", "LAW", "uk-2002"),  # SPLA timeout
]


_n = len(GRAPHS)
for _i, (stem, grp, mtx) in enumerate(GRAPHS, 1):
    dest = OUT / f"{stem}.mtx"
    if dest.is_file():
        print(f"[{_i}/{_n}] skip {stem}")
        continue
    print(f"[{_i}/{_n}] download {stem} …")
    url = f"{BASE}/{grp}/{mtx}.tar.gz"
    with urllib.request.urlopen(url, timeout=600, context=_SSL) as r:
        data = r.read()
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tf:
        members = [m for m in tf.getmembers() if m.name.endswith(".mtx")]
        m = max(members, key=lambda x: x.size or 0)
        dest.write_bytes(tf.extractfile(m).read())
    print(f"[{_i}/{_n}] ok {stem}")

[1/5] skip amazon0601
[2/5] skip citationCiteseer
[3/5] download coPapersDBLP …
[3/5] ok coPapersDBLP
[4/5] download in-2004 …
[4/5] ok in-2004
[5/5] download cit-Patents …
[5/5] ok cit-Patents


In [30]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.io import mmread, mmwrite
from scipy.sparse import coo_matrix, csgraph, issparse


def mm_hdr(path):
    hdr, n, m = None, None, None
    with open(path, encoding="latin-1", errors="replace") as f:
        for ln in f:
            s = ln.strip()
            if s.startswith("%%MatrixMarket"):
                hdr = s.split()
            elif s and not s.startswith("%"):
                n, m = map(int, s.replace(",", " ").split()[:2])
                break
    return n, m, hdr[-2].lower(), hdr[-1].lower()


def symmetrize_min(A):
    A = A.tocoo()
    n, best = A.shape[0], {}
    for i, j, v in zip(A.row, A.col, A.data):
        if i == j:
            continue
        a, b = (i, j) if i < j else (j, i)
        t = best.get((a, b))
        if t is None or v < t:
            best[(a, b)] = v
    if not best:
        return coo_matrix((n, n)).tocsr()
    rs, cs, ds = [], [], []
    for (a, b), v in best.items():
        rs += (a, b)
        cs += (b, a)
        ds += (v, v)
    return coo_matrix((ds, (rs, cs)), shape=(n, n)).tocsr()


def graph_table(stems: list[str]):
    rows = []
    for stem in stems:
        p = OUT / f"{stem}.mtx"
        n, m, fld, sym = mm_hdr(p)
        u = sym in ("symmetric", "hermitian")
        A = mmread(p).tocsr()
        A.eliminate_zeros()
        co = A.tocoo()
        r, c, d = co.row, co.col, co.data
        E = int(((r < c) | ((r == c) & (d != 0))).sum()) if u else int((d != 0).sum())
        B = A.astype(bool) if u else (A + A.T).astype(bool)
        comp = csgraph.connected_components(B, directed=False)[0]
        if fld == "pattern":
            wflag = "pattern"
        elif np.iscomplexobj(d):
            wflag = "yes" if np.all(np.real(d) >= 0) else "no"
        else:
            wflag = "yes" if np.all(d >= 0) else "no"
        rows.append(
            {
                "file": p.name,
                "kind": "undirected" if u else "directed",
                "V": n,
                "E": E,
                "components": comp,
                "nonneg_weights": wflag,
            }
        )
    return pd.DataFrame(rows)


display(graph_table([g[0] for g in GRAPHS]))

,file,kind,V,E,components,nonneg_weights
0,amazon0601.mtx,directed,403394,3387388,7,pattern
1,citationCiteseer.mtx,undirected,268495,1156647,1,pattern
2,coPapersDBLP.mtx,undirected,540486,15245729,1,pattern
3,in-2004.mtx,directed,1382908,16917053,134,pattern
4,cit-Patents.mtx,directed,3774768,16518948,3627,pattern


In [31]:
GRAPHS_BENCH: list[str] = []

for stem, *_ in GRAPHS:
    p = OUT / f"{stem}.mtx"
    row = graph_table([stem]).iloc[0]
    if row["nonneg_weights"] == "no":
        continue
    _, _, fld, sym = mm_hdr(p)
    undir = sym in ("symmetric", "hermitian")
    raw = mmread(p)
    co = raw.tocoo() if issparse(raw) else coo_matrix(raw)
    data = np.ones(co.nnz, np.float64) if fld == "pattern" else np.asarray(co.data, np.float64)
    A = coo_matrix((data, (co.row, co.col)), shape=co.shape).tocsr()
    A.eliminate_zeros()
    if not undir:
        A = symmetrize_min(A)
        A.eliminate_zeros()
        if A.nnz and np.any(A.data < 0):
            continue
        mmwrite(str(p), A, field="real", symmetry="symmetric")
    elif fld == "pattern":
        mmwrite(str(p), A, field="real", symmetry="symmetric")
    GRAPHS_BENCH.append(stem)

display(graph_table(GRAPHS_BENCH))

,file,kind,V,E,components,nonneg_weights
0,amazon0601.mtx,undirected,403394,2443408,7,yes
1,citationCiteseer.mtx,undirected,268495,1156647,1,yes
2,coPapersDBLP.mtx,undirected,540486,15245729,1,yes
3,in-2004.mtx,undirected,1382908,13591473,134,yes
4,cit-Patents.mtx,undirected,3774768,16518947,3627,yes


In [35]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

RESULTS = (Path(".") / "results").resolve()
RESULTS.mkdir(parents=True, exist_ok=True)
SPLA = (Path("..") / "boruvka" / "boruvka_spla" / "build" / "boruvka_spla").resolve()
SPARK = (Path("..") / "boruvka" / "boruvka_spark").resolve()

N_TIMED, N_WARMUP = 15, 5


def _spark_cmd(mtx: Path, csv_out: Path):
    q = lambda p: shlex.quote(str(p))
    inner = f"cd {q(SPARK)} && make run GRAPH={q(mtx)} CSV={q(csv_out)} RUNS={N_TIMED} WARMUP={N_WARMUP}"
    sh = os.environ.get("SHELL") or ("/bin/zsh" if sys.platform == "darwin" else "/bin/bash")
    fl = "-ilc" if Path(sh).name == "zsh" else "-lc"
    return [sh, fl, inner]


def _agg_time(df: pd.DataFrame):
    c = "time_ms" if "time_ms" in df.columns else "mean_time_ms"
    t = pd.to_numeric(df[c], errors="coerce").dropna()
    n = len(t)
    std = float(t.std(ddof=1)) if n > 1 else 0.0
    q1, q3 = float(t.quantile(0.25)), float(t.quantile(0.75))
    return float(t.mean()), std, float(t.median()), q3 - q1


def _mst_meta(df: pd.DataFrame):
    r = df.iloc[-1]
    w = float(r["mst_weight"])
    w = int(w) if w == int(w) else w
    return w, int(r["vertices"]), int(r["mst_edges"])


def bench(impl: str) -> pd.DataFrame:
    rows = []
    for stem in GRAPHS_BENCH:
        mtx = (OUT / f"{stem}.mtx").resolve()
        csv_out = (RESULTS / f"{impl}-{stem}.csv").resolve()
        csv_out.unlink(missing_ok=True)
        if impl == "spla":
            subprocess.run(
                [str(SPLA), "--mtxpath", str(mtx), "--out", str(csv_out), "--niters", str(N_TIMED), "--warmup", str(N_WARMUP)],
            )
        else:
            csv_out.parent.mkdir(parents=True, exist_ok=True)
            r = subprocess.run(_spark_cmd(mtx, csv_out), capture_output=True, text=True)
            if r.returncode != 0:
                print((r.stderr or r.stdout or "").strip())
        df = pd.read_csv(csv_out)
        mn, sd, med, iqr = _agg_time(df)
        mw, mv, me = _mst_meta(df)
        rows.append(
            {
                "graph": stem,
                "library": impl,
                "time_mean_ms": mn,
                "time_std_ms": sd,
                "time_median_ms": med,
                "time_iqr_ms": iqr,
                "mst_weight": mw,
                "mst_vertices": mv,
                "mst_edges": me,
            }
        )
    return pd.DataFrame(
        rows,
        columns=[
            "graph",
            "library",
            "time_mean_ms",
            "time_std_ms",
            "time_median_ms",
            "time_iqr_ms",
            "mst_weight",
            "mst_vertices",
            "mst_edges",
        ],
    )


bench_spla = bench("spla")
bench_spark = bench("spark")
display(bench_spla)
display(bench_spark)

Python(61755) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


env: OpenCL Acc Apple device: Apple M1 Pro vendor: mcu:16 wave:8 mwgs:256


Python(61763) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


env: OpenCL Acc Apple device: Apple M1 Pro vendor: mcu:16 wave:8 mwgs:256


Python(61782) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


...

Python(61809) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


env: OpenCL Acc Apple device: Apple M1 Pro vendor: mcu:16 wave:8 mwgs:256


Python(61860) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62079) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62439) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(62708) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(63580) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


,graph,library,time_mean_ms,time_std_ms,time_median_ms,time_iqr_ms,mst_weight,mst_vertices,mst_edges
0,amazon0601,spla,184.555800,1.952701,184.710,2.9495,403387,403394,403387
1,citationCiteseer,spla,346.769333,5.875149,345.398,6.1340,268494,268495,268494
2,coPapersDBLP,spla,1264.860133,24.323113,1266.131,40.4485,540485,540486,540485
3,in-2004,spla,1795.434667,116.491765,1765.543,19.0430,1382774,1382908,1382774
4,cit-Patents,spla,11413.122200,102.147039,11431.952,101.2760,3771141,3774768,3771141


,graph,library,time_mean_ms,time_std_ms,time_median_ms,time_iqr_ms,mst_weight,mst_vertices,mst_edges
0,amazon0601,spark,7401.445333,90.975834,7436.71,145.620,403387,403394,403387
1,citationCiteseer,spark,7542.159333,164.966872,7517.12,242.215,268494,268495,268494
2,coPapersDBLP,spark,13649.022667,575.038601,13473.92,595.085,540485,540486,540485
3,in-2004,spark,38139.948667,1842.231898,38079.82,2807.520,1382774,1382867,1382774
4,cit-Patents,spark,133338.458000,4495.966797,132294.36,7301.075,3771141,3774768,3771141
